# A1 · SQL basics

**Question:** which customer market segment spent the most in 1996, and how many orders was that?

Work top to bottom with `Shift+Enter`. Every `%%sql` cell sends its SQL to **Trino**, the lab's shared SQL engine, **as you**. The lesson (`README.md` in this folder) explains each step in more detail.

## 0. Connect

This cell loads JupySQL (the `%%sql` cells), connects it to Trino with your own login, and works out the name of **your schema**, the one place where you can create tables. You never type a password: `lakehouse` gets your login token from JupyterHub.

In [ ]:
import lakehouse

%load_ext sql
%config SqlMagic.displaylimit = 20
trino = lakehouse.sql_engine()
%sql trino --alias trino

me = lakehouse.whoami()
schema = f"dbt_{me}"
print(f"You are {me}. Your schema is lakehouse.{schema}")

Expected: `You are <your user name>. Your schema is lakehouse.dbt_<your user name>`.

In the cells below, `{{schema}}` is replaced by your schema name before the SQL is sent.

## 1. Look before you ask

`catalog.schema.table`: `lakehouse` is the catalog, `samples` the schema, `customer` the table. `LIMIT 5` keeps the answer small.

In [ ]:
%%sql
SELECT *
FROM lakehouse.samples.customer
LIMIT 5

Expected: 5 rows with the columns `custkey, name, address, nationkey, phone, acctbal, mktsegment, comment`.

## 2. Choose columns and filter rows: `SELECT` and `WHERE`

Text values go in single quotes. `ORDER BY acctbal DESC` sorts the richest first.

In [ ]:
%%sql
SELECT name, mktsegment, acctbal
FROM lakehouse.samples.customer
WHERE mktsegment = 'BUILDING' AND acctbal > 9000
ORDER BY acctbal DESC
LIMIT 10

**Try it:** change `'BUILDING'` to `'MACHINERY'`, or `9000` to `9900`, and run the cell again.

## 3. Count and summarize: `GROUP BY`

Every column in `SELECT` must be either in `GROUP BY` or inside an aggregate (`count`, `sum`, `avg`, `min`, `max`).

In [ ]:
%%sql
SELECT mktsegment, count(*) AS customers
FROM lakehouse.samples.customer
GROUP BY mktsegment
ORDER BY customers DESC

Expected: 5 segments, each with roughly 300 customers (BUILDING has the most, 337).

## 4. Combine tables: `JOIN`

Orders know the customer key (`custkey`), customers know the segment. `ON o.custkey = c.custkey` says which rows belong together.

In [ ]:
%%sql
SELECT o.orderkey, o.orderdate, o.totalprice, c.name, c.mktsegment
FROM lakehouse.samples.orders o
JOIN lakehouse.samples.customer c ON o.custkey = c.custkey
LIMIT 5

## 5. Filter on dates

"From Jan 1st 1996, and before Jan 1st 1997" means "the year 1996".

In [ ]:
%%sql
SELECT count(*) AS orders_1996
FROM lakehouse.samples.orders
WHERE orderdate >= DATE '1996-01-01' AND orderdate < DATE '1997-01-01'

Expected: `2297`.

## 6. Exercise: answer the question and save it

First make sure your schema exists (safe to run again):

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS lakehouse.{{schema}}

Now complete the query: join orders to customers, keep only 1996 orders, and group by segment. Replace each `...`, then run the cell. The columns must be called `market_segment`, `orders` and `total_price`.

`CREATE OR REPLACE TABLE ... AS SELECT` stores the result as a table; `OR REPLACE` lets you run it again after a fix.

In [ ]:
%%sql
CREATE OR REPLACE TABLE lakehouse.{{schema}}.a1_segment_revenue_1996 AS
SELECT
    c.mktsegment                 AS market_segment,
    count(*)                     AS orders,
    round(sum(o.totalprice), 2)  AS total_price
FROM lakehouse.samples.orders o
JOIN ...
WHERE ...
GROUP BY ...

Look at your table:

In [ ]:
%%sql
SELECT * FROM lakehouse.{{schema}}.a1_segment_revenue_1996 ORDER BY total_price DESC

Expected:

| market_segment | orders | total_price |
|---|---|---|
| BUILDING | 557 | 80313228.33 |
| FURNITURE | 472 | 65238850.11 |
| AUTOMOBILE | 452 | 61700951.28 |
| HOUSEHOLD | 416 | 61550669.49 |
| MACHINERY | 400 | 55680541.33 |

## 7. Check your work

This runs the module's checkpoint as you. It looks only at the table, not at your SQL. If something is not right yet, it tells you what it found and what to try.

In [ ]:
!lab-tracks check A1

If `lab-tracks` is not found, run `!python3 checkpoint.py` instead: it is the same check.

**Also on the `full` profile?** Open Superset → **SQL** → **SQL Lab**, pick the database **Lakehouse (Trino)**, and run the queries from steps 1 to 5 there too. Type `dbt_<your user name>` where the notebook says `{{schema}}`.

**Next:** A2 · exploratory analysis with DuckDB and JupySQL.